# Crypto Price Forecast - Analysis & Visualization
This notebook is used for importing data and models from the src package, analyzing the data, and visualizing the results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.feature_engineering import engineer_features

In [ ]:
# Load data
df = load_data('data/crypto_statistics_data.csv')
df.head()

In [ ]:
# Plotting Close Price over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='date', y='close')
plt.title('Crypto Close Price')
plt.show()

In [ ]:
# Reproducibility and imports
import os, random
import numpy as np
import tensorflow as tf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"numpy {np.__version__}, pandas {pd.__version__}, tensorflow {tf.__version__}")

In [ ]:
# Dataset overview & quick EDA
try:
	df
except NameError:
	try:
		from src.data_loader import load_data
		df = load_data('data/crypto_statistics_data.csv')
	except Exception:
		df = pd.read_csv('data/crypto_statistics_data.csv', parse_dates=['date'])

df['date'] = pd.to_datetime(df['date'])
print('Shape:', df.shape)
print(df.dtypes)
print('Missing values:\n', df.isnull().sum())
print('Date range:', df['date'].min(), 'to', df['date'].max())

fig = px.line(df, x='date', y='close', title='Close Price over Time')
fig.update_layout(width=900, height=400)
fig.show()

In [ ]:
# Feature engineering snapshot and indicators
eng = engineer_features(df.copy())
cols = ['date', 'close']
if 'EMA_12' in eng.columns:
	ema = 'EMA_12'
else:
	ema = [c for c in eng.columns if 'EMA' in c][:1]
if isinstance(ema, list):
	ema = ema[0] if ema else None

rsi_col = 'RSI_14' if 'RSI_14' in eng.columns else None
macd_col = 'MACD' if 'MACD' in eng.columns else None

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
					row_heights=[0.7, 0.3],
					vertical_spacing=0.05,
					subplot_titles=('Price & EMA', 'RSI'))
fig.add_trace(go.Scatter(x=eng['date'], y=eng['close'], name='Close'), row=1, col=1)
if ema:
	fig.add_trace(go.Scatter(x=eng['date'], y=eng[ema], name=ema), row=1, col=1)
if rsi_col:
	fig.add_trace(go.Scatter(x=eng['date'], y=eng[rsi_col], name=rsi_col, line=dict(color='orange')), row=2, col=1)
fig.update_layout(height=600, width=900, showlegend=True, title_text='Engineered Indicators Snapshot')
fig.show()

In [ ]:
# Preprocessing, scaling, and sequence preview
series = eng['close'].values.reshape(-1,1)
split = int(len(series)*0.8)
train, test = series[:split], series[split:]

scaler = MinMaxScaler()
train_s = scaler.fit_transform(train)
test_s = scaler.transform(test)

LOOKBACK = 30
def make_sequence(arr, lookback):
	X, y = [], []
	for i in range(len(arr)-lookback):
		X.append(arr[i:i+lookback,0])
		y.append(arr[i+lookback,0])
	return np.array(X), np.array(y)

X_train, y_train = make_sequence(train_s, LOOKBACK)
X_test, y_test = make_sequence(np.vstack([train_s[-LOOKBACK:], test_s]), LOOKBACK)

print('Train shapes:', X_train.shape, y_train.shape)
print('Test shapes:', X_test.shape, y_test.shape)

In [ ]:
# Load saved models and compare on test set
from tensorflow.keras.models import load_model
results = []

for name, path in [('LSTM','results/lstm_model.keras'), ('GRU','results/gru_model.keras')]:
	try:
		m = load_model(path)
		# prepare X_test in model input shape (samples, lookback, 1)
		X_in = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
		preds_s = m.predict(X_in, verbose=0).ravel()
		preds = scaler.inverse_transform(preds_s.reshape(-1,1)).ravel()
		y_true = scaler.inverse_transform(y_test.reshape(-1,1)).ravel()
		rmse = mean_squared_error(y_true, preds, squared=False)
		mae = mean_absolute_error(y_true, preds)
		mape = (np.mean(np.abs((y_true - preds) / y_true)))*100
		results.append({'model':name, 'rmse':rmse, 'mae':mae, 'mape':mape})
	except Exception as e:
		print(f"Could not load {name} at {path}: {e}")

if results:
	res_df = pd.DataFrame(results).sort_values('rmse')
	display(res_df)
else:
	print('No models loaded; ensure results/*.keras exist.')

In [ ]:
# Predictions vs Actual and residuals visualization
import numpy as np
fig = make_subplots(rows=2, cols=1, subplot_titles=('Predictions vs Actual','Residuals'), row_heights=[0.7,0.3], shared_xaxes=False)

# combine predictions if available
pred_dfs = {}
for name, path in [('LSTM','results/lstm_model.keras'), ('GRU','results/gru_model.keras')]:
	try:
		m = load_model(path)
		X_in = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
		preds_s = m.predict(X_in, verbose=0).ravel()
		preds = scaler.inverse_transform(preds_s.reshape(-1,1)).ravel()
		pred_dfs[name] = preds
		fig.add_trace(go.Scatter(y=preds, name=f'{name} Pred'), row=1, col=1)
	except:
		pass

# actual
y_true = scaler.inverse_transform(y_test.reshape(-1,1)).ravel()
fig.add_trace(go.Scatter(y=y_true, name='Actual', line=dict(color='black')), row=1, col=1)

# residuals histogram for best model if exists
if pred_dfs:
	best = min(pred_dfs.keys(), key=lambda k: mean_squared_error(y_true, pred_dfs[k], squared=False))
	res = y_true - pred_dfs[best]
	fig.add_trace(go.Histogram(x=res, name='Residuals'), row=2, col=1)
	# uncertainty band (mean +/- std across models)
	preds_stack = np.vstack(list(pred_dfs.values()))
	mean_pred = preds_stack.mean(axis=0)
	std_pred = preds_stack.std(axis=0)
	fig.add_trace(go.Scatter(y=mean_pred + std_pred, name='Upper band', line=dict(color='rgba(0,0,0,0.2)'), showlegend=True), row=1, col=1)
	fig.add_trace(go.Scatter(y=mean_pred - std_pred, name='Lower band', line=dict(color='rgba(0,0,0,0.2)'), showlegend=True), row=1, col=1)
else:
	fig.add_annotation(text='No predictions available', x=0.5, y=0.5, showarrow=False, row=1, col=1)

fig.update_layout(height=800, width=900, showlegend=True)
fig.show()